In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn import preprocessing
from urllib.request import urlretrieve
import plotly.express as px
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
%matplotlib inline

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',150)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size']=14
matplotlib.rcParams['figure.figsize']=(10,6)
matplotlib.rcParams['figure.facecolor']='#00000000'

In [ ]:
selected_cols=['fare_amount',	'pickup_datetime', 'pickup_longitude',	'pickup_latitude',	'dropoff_longitude',	'dropoff_latitude',	'passenger_count']
selected_cols

In [ ]:
dtypes = {
	'fare_amount': 'float32',
	'pickup_longitude': 'float32',
	'pickup_latitude': 'float32',
	'dropoff_longitude': 'float32',
	'dropoff_latitude': 'float32',
	'passenger_count': 'uint8'
}
dtypes

In [ ]:
import random
sample_fraction = 0.01
def skip_row(row_idx):
    if row_idx==0:
        return False
    return random.random()>sample_fraction

random.seed(42)
df=pd.read_csv('train.csv',usecols=selected_cols,dtype=dtypes, parse_dates=['pickup_datetime'], skiprows=skip_row)
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df['pickup_datetime'].min(),df['pickup_datetime'].max()

In [ ]:
test_df=pd.read_csv('test.csv',dtype=dtypes,parse_dates=['pickup_datetime'])
test_df

In [ ]:
test_df.info()

In [ ]:
test_df.describe()

In [ ]:
test_df.isna().sum()

In [ ]:
test_df['pickup_datetime'].min(),test_df['pickup_datetime'].max()


****** 
***Preprocessing***

In [ ]:
from sklearn.model_selection import train_test_split
train_df,val_df=train_test_split(df,test_size=0.2,random_state=42)

In [ ]:
len(train_df),len(val_df)

**filling missing data**

In [ ]:
train_df=train_df.dropna()
val_df=val_df.dropna()

**extract inpust and targets from datasets**

In [ ]:
input_cols=['pickup_longitude',	'pickup_latitude',	'dropoff_longitude',	'dropoff_latitude',	'passenger_count']
target_cols='fare_amount'

**training and validation inputs and targets**

In [ ]:
train_inputs=train_df[input_cols]
train_targets=train_df[target_cols]
val_inputs=val_df[input_cols]
val_targets=val_df[target_cols]
test_inputs=test_df[input_cols]

In [ ]:
train_inputs

In [ ]:
train_targets

In [ ]:
val_inputs

In [ ]:
val_targets

In [ ]:
test_inputs

****** 
***Train and evaluate Hardcoded Model***

In [ ]:
class MeanRegressor:
    def fit(self,inputs,targets):
        self.mean=targets.mean()
    def predict(self,inputs):
        return np.full(inputs.shape[0],self.mean)

In [ ]:
np.full(10,3)

In [ ]:
mean_model=MeanRegressor()
mean_model.fit(train_inputs,train_targets)

In [ ]:
mean_model.mean

In [ ]:
train_Pred=mean_model.predict(train_inputs)
train_Pred

In [ ]:
val_Pred=mean_model.predict(val_inputs)
val_Pred

In [ ]:
from sklearn.metrics import root_mean_squared_error
def rmse(targetts,preds):
    return root_mean_squared_error(targetts,preds)

In [ ]:
print('RMSE of Training model: ',rmse(train_targets,train_Pred))
print('RMSE of Validation model: ',rmse(val_targets,val_Pred))

****** 
***Train and evaluate Baseline Model***

In [ ]:
from sklearn.linear_model import LinearRegression
lr_model=LinearRegression().fit(train_inputs,train_targets)

In [ ]:
train_pred=lr_model.predict(train_inputs)
val_Pred=lr_model.predict(val_inputs)
train_Pred,val_Pred

In [ ]:
print('RMSE of Training model: ',rmse(train_targets,train_Pred))
print('RMSE of Validation model: ',rmse(val_targets,val_Pred))

****** 
***Predict and Submit Models***

In [ ]:
def predict_and_submit(model,test_inputs,fname):
    test_preds=model.predict(test_inputs)
    sub_df=pd.read_csv('sample_submission.csv')
    sub_df['fare_amount']=test_preds
    sub_df.to_csv(fname+'.csv',index=None)
    return sub_df

****** 
***Feature Engineering***

**Adding date specific features**

In [ ]:
def add_dateparts(df,col):
 df[col+'_year']=df[col].dt.year
 df[col+'_month']=df[col].dt.month
 df[col+'_day']=df[col].dt.day
 df[col+'_weekday']=df[col].dt.weekday
 df[col+'_hour']=df[col].dt.hour

In [ ]:
col='pickup_datetime'
df[col].dt.year

In [ ]:
add_dateparts(train_df,'pickup_datetime')
add_dateparts(val_df,'pickup_datetime')
add_dateparts(test_df,'pickup_datetime')

In [ ]:
train_df

In [ ]:
val_df

In [ ]:
test_df

**Adding distance feature**

In [ ]:
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    
    All args must be of equal length.    
    
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6378.137 * c
    return km

In [ ]:
def add_trip_distance(df):
    df['trip_distance']=haversine_np(df['pickup_longitude'],df['pickup_latitude'],df['dropoff_longitude'],df['dropoff_latitude'])

In [ ]:
add_trip_distance(train_df)
add_trip_distance(val_df)
add_trip_distance(test_df)

In [ ]:
train_df

In [ ]:
val_df

In [ ]:
test_df

**Distance from Popular landmark**

In [ ]:
jfk_lonlat=-73.7781,40.6413
lga_lonlat=-73.8740,40.7769
ewr_lonlat=-74.1745,40.6895
met_lonlat=-73.9632,40.7794
wtc_lonlat=-74.0099,40.7126

In [ ]:
def add_landmark_distance(df,landmark_name,landmark_lonlat):
    lan,lot=landmark_lonlat
    df[landmark_name+'_drop_distance']=haversine_np(lan,lot,df['dropoff_longitude'],df['dropoff_latitude'])

In [ ]:
def add_landmark(df):
    landmarks=[('jfk',jfk_lonlat),('lga',lga_lonlat),('ewr',ewr_lonlat),('met',met_lonlat),('wtc',wtc_lonlat)]
    for name,lanlot in landmarks:
        add_landmark_distance(df,name,lanlot)

In [ ]:
# add_landmark_distance(train_df,'JFK_Airport',jfk_lonlat)
# add_landmark_distance(train_df,'LGA_Airport',lga_lonlat)
# add_landmark_distance(train_df,'EWR_Airport',ewr_lonlat)
# add_landmark_distance(train_df,'MET_Meuseum',met_lonlat)
# add_landmark_distance(train_df,'World_trade_center',wtc_lonlat)

# add_landmark_distance(val_df,'JFK_Airport',jfk_lonlat)
# add_landmark_distance(val_df,'LGA_Airport',lga_lonlat)
# add_landmark_distance(val_df,'EWR_Airport',ewr_lonlat)
# add_landmark_distance(val_df,'MET_Meuseum',met_lonlat)
# add_landmark_distance(val_df,'World_trade_center',wtc_lonlat)

# add_landmark_distance(test_df,'JFK_Airport',jfk_lonlat)
# add_landmark_distance(test_df,'LGA_Airport',lga_lonlat)
# add_landmark_distance(test_df,'EWR_Airport',ewr_lonlat)
# add_landmark_distance(test_df,'MET_Meuseum',met_lonlat)
# add_landmark_distance(test_df,'World_trade_center',wtc_lonlat)


add_landmark(train_df)
add_landmark(val_df)
add_landmark(test_df)

In [ ]:
train_df

In [ ]:
val_df

In [ ]:
test_df

****** 
***Removing Outliers and invalid data***

fare_amount: $1 to 500\
longitude: -75 to -72\
lattitude: 40 to 42\
passenger_count: 1 to 6

In [ ]:
def remove_outliers(df):
    return df[(df['fare_amount'] >= 1.)& 
              (df['fare_amount'] <= 500.) & 
              (df['pickup_longitude'] >= -75) & 
              (df['pickup_longitude'] <= -72) & 
              (df['pickup_latitude'] >= 40) & 
              (df['pickup_latitude'] <= 42) & 
              (df['dropoff_longitude'] >= -75) & 
              (df['dropoff_longitude'] <= -72) & 
              (df['dropoff_latitude'] >= 40) & 
              (df['dropoff_latitude'] <= 42) & 
              (df['passenger_count'] >= 1) & 
              (df['passenger_count'] <= 6)]

In [ ]:
train_df=remove_outliers(train_df)
val_df=remove_outliers(val_df)

In [ ]:
train_df.describe()

In [ ]:
train_df.to_parquet('train.parquet')
val_df.to_parquet('val.parquet')
test_df.to_parquet('test.parquet')

****** 
***Training and Evaluate models***

In [ ]:
train_df.columns

In [ ]:
input_cols=['pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'pickup_datetime_year', 'pickup_datetime_month', 'pickup_datetime_day',
       'pickup_datetime_weekday', 'pickup_datetime_hour', 'trip_distance',
       'jfk_drop_distance', 'lga_drop_distance', 'ewr_drop_distance',
       'met_drop_distance', 'wtc_drop_distance']
target_cols='fare_amount'

In [ ]:
train_inputs=train_df[input_cols]
train_targets=train_df[target_cols]
val_inputs=val_df[input_cols]
val_targets=val_df[target_cols]
test_inputs=test_df[input_cols]

In [ ]:
val_inputs

In [ ]:
def eval_model(model):
    train_pred=model.predict(train_inputs)
    val_pred=model.predict(val_inputs)
    train_rmse=rmse(train_targets,train_pred)
    val_rmse=rmse(val_targets,val_pred)
    return train_rmse,val_rmse,train_pred,val_pred

**Ridge Regression**

In [ ]:
from sklearn.linear_model import Ridge
model_ridge=Ridge(random_state=42,alpha=0.9).fit(train_inputs,train_targets)
train_rmse,val_rmse,train_pred,val_pred=eval_model(model_ridge)
print('train rmse: ',train_rmse)
print('val rmse: ',val_rmse)
print('train pred: ',train_Pred)
print('val pred: ',val_Pred)

**Saving the submission**

In [ ]:
predict_and_submit(model=model_ridge,test_inputs=test_inputs,fname='Ridge_submissions')

**Random Forest**

In [ ]:
%%time
from sklearn.ensemble import RandomForestRegressor
model_rf=RandomForestRegressor(random_state=42,n_jobs=-1,max_depth=10,n_estimators=100).fit(train_inputs,train_targets)

In [ ]:
train_rmse,val_rmse,train_pred,val_pred=eval_model(model_rf)
print('train rmse: ',train_rmse)
print('val rmse: ',val_rmse)
print('train pred: ',train_Pred)
print('val pred: ',val_Pred)

**Saving the submission**

In [ ]:
predict_and_submit(model=model_rf,test_inputs=test_inputs,fname='rf_submissions')

**Gradient Boosting**

In [ ]:
from xgboost import XGBRegressor
model_xgb=XGBRegressor(max_depth=5,objective='reg:squarederror',n_estimators=200,random_state=42,n_jobs=-1).fit(train_inputs,train_targets)

In [ ]:
train_rmse,val_rmse,train_pred,val_pred=eval_model(model_xgb)
print('train rmse: ',train_rmse)
print('val rmse: ',val_rmse)
print('train pred: ',train_Pred)
print('val pred: ',val_Pred)

**Saving the submission**

In [ ]:
predict_and_submit(model=model_xgb,test_inputs=test_inputs,fname='xgb_submissions')

**Tune Hyperparameters**

In [ ]:
def params_testing(model,**params):
    model=model(**params).fit(train_inputs,train_targets)
    train_rmse,val_rmse,train_pred,val_pred=eval_model(model)
    return train_rmse,val_rmse

def params_tuning_plot(model,params_name,params_values,fixed_params=None):
    if fixed_params is None:
        fixed_params={}
    train_error=[]
    val_error=[]
    for value in params_values:
        params=fixed_params.copy()
        params[params_name]=value
        train_rmse,val_rmse=params_testing(model,**params)
        train_error.append(train_rmse)
        val_error.append(val_rmse)
    
    plt.figure(figsize=(10,15))
    plt.title(f'{model.__name__} {params_name} tuning')
    plt.plot(params_values, train_error, 'b-o', label='train error', markersize=5)
    plt.plot(params_values, val_error, 'r-o', label='val error', markersize=5)
    plt.xlabel(params_name)
    plt.ylabel('RMSE')
    plt.legend()

best_params={
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'learning_rate':0.05
}

In [ ]:
params_tuning_plot(XGBRegressor,'num_estimators',[10,20,50,100,200,300,400,500],best_params)

In [ ]:
params_tuning_plot(XGBRegressor,'max_depth',[3,5,7,9,11,15],best_params)

In [ ]:
params_tuning_plot(XGBRegressor,'max_depth',[5,6,7,8,9,10,11,12,13],best_params)

In [ ]:
best_params={
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9
}
params_tuning_plot(XGBRegressor,'learning_rate',[0.02,0.05,0.07,0.1,0.13,0.15,0.17,0.2,0.25],best_params)

In [ ]:
best_params={
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9
}
params_tuning_plot(XGBRegressor,'learning_rate',[0.15,0.16,0.17,0.18,0.19,0.2],best_params)
params_tuning_plot(XGBRegressor,'learning_rate',[0.2,0.21,0.22,0.23,0.24,0.25],best_params)

In [ ]:
best_params={
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9,
    'learning_rate':0.16
}
params_tuning_plot(XGBRegressor,'sub_sample',[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],best_params)

In [ ]:
best_params={
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9,
    'learning_rate':0.16,
    'sub_sample':0.8
}
params_tuning_plot(XGBRegressor,'colsample_bytree',[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],best_params)

In [ ]:
best_params={
    'num_estimators':200,
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9,
    'learning_rate':0.16,
    'sub_sample':0.8,
    'colsample_bytree':0.9
}
params_tuning_plot(XGBRegressor,'max_depth',[3,5,7,9,11,15],best_params)

In [ ]:
best_params={
    'num_estimators':200,
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':9,
    'learning_rate':0.16,
    'sub_sample':0.8,
    'colsample_bytree':0.9
}
params_tuning_plot(XGBRegressor,'max_depth',[6,7,8,9,10,11],best_params)

In [ ]:
best_params={
    'num_estimators':200,
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':8,
    'learning_rate':0.16,
    'sub_sample':0.8,
    'colsample_bytree':0.9
}
params_tuning_plot(XGBRegressor,'learning_rate',[0.10,0.11,0.12,0.13,0.14,0.15,0.16,0.17,0.18,0.19,0.2,0.21,0.22,0.23,0.24,0.25],best_params)

In [ ]:
best_params={
    'num_estimators':200,
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':8,
    'learning_rate':0.15,
    'sub_sample':0.8,
    'colsample_bytree':0.9
}
params_tuning_plot(XGBRegressor,'sub_sample',[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],best_params)

In [ ]:
best_params={
    'num_estimators':200,
    'random_state':42,
    'n_jobs':-1,
    'objective':'reg:squarederror',
    'max_depth':8,
    'learning_rate':0.15,
    'sub_sample':0.8,
    'colsample_bytree':0.9
}
params_tuning_plot(XGBRegressor,'colsample_bytree',[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],best_params)

**Final XGBoost Model**

In [ ]:
%%time
model_finalXGB=XGBRegressor(
    num_estimators=200, 
    random_state=42, 
    n_jobs=-1, 
    objective='reg:squarederror', 
    max_depth=8, 
    learning_rate=0.15, 
    sub_sample=0.8, 
    colsample_bytree=0.9
).fit(train_inputs,train_targets)

In [ ]:
train_rmse,val_rmse,train_pred,val_pred=eval_model(model_finalXGB)
print('train rmse: ',train_rmse)
print('val rmse: ',val_rmse)
print('train pred: ',train_Pred)
print('val pred: ',val_Pred)

**Saving the submission**

In [ ]:
predict_and_submit(model=model_finalXGB,test_inputs=test_inputs,fname='final_xgb_submissions')